# 03. 학습 — 모델 · 하이퍼파라미터 · 실험 관리

|  |  |
|---|---|
| **입력** | `data/pill_yolo/data.yaml` (02 생성) 또는 `data/pill_yolo_base/data.yaml` (01 생성) |
| **출력** | `runs/<실험명>/weights/best.pt`, `experiments/log.jsonl` |

---

## 실험 규칙

**① 한 번에 하나만 바꾸기**
증강과 모델을 동시에 바꾸면 뭐가 효과였는지 알 수 없습니다.

**② 학습 예산 통일**
`EPOCHS=100, PATIENCE=15` 를 팀 표준으로 고정하세요.
"VGG 30에포크 vs YOLO 200에포크" 로 나온 표는 아무것도 증명하지 못합니다.
대신 **실제로 몇 에포크에 멈췄는지**를 기록하면 그 자체가 좋은 결과가 됩니다.

**③ 변경점을 `NOTE` 에 남기기**
`experiments/log.jsonl` 에 자동 누적되어 `04_evaluation` 에서 표로 만듭니다.

---

## ★ 온라인 증강을 기본으로 끄는 이유

02 에서 **이미 오프라인으로 증강을 끝냈습니다** (기하 3배 + Copy&Paste).
여기서 Ultralytics 내장 증강을 또 걸면 **이중 증강**이 됩니다.

특히 위험한 것:

| 내장 인자 | 왜 끄는가 |
|---|---|
| `hsv_h`, `hsv_s` | `color_class1`(색상)이 클래스 정보인데 색을 또 흔듭니다 |
| `degrees` | 02 의 `RotateScale` 이 내접 타원으로 정확히 처리했는데 다시 돌리면 박스가 부풉니다 |
| `fliplr`, `flipud` | 각인(`RE20`, `ATV`)이 뒤집힙니다 |
| `copy_paste` | 02 의 Copy&Paste 와 중복. 게다가 내장 버전은 세그멘테이션 마스크가 필요합니다 |
| `erasing` | 각인이 통째로 지워질 수 있습니다 |

`USE_ONLINE_AUG = True` 로 두면 온라인 증강을 켤 수 있습니다.
**증강 없는 데이터셋(`pill_yolo_base`)으로 실험할 때만** 켜세요.

## 증강 효과를 정량화하는 3-실험 세트

| 실험 | `DATA_YAML` | `USE_ONLINE_AUG` | 보는 것 |
|---|---|---|---|
| `exp_base` | `pill_yolo_base` | `False` | 증강 전혀 없음 (바닥) |
| `exp_online` | `pill_yolo_base` | `True` | 온라인 증강만 |
| `exp_offline` | `pill_yolo` | `False` | ★ 02 의 기하 3배 + Copy&Paste |

세 실험을 같은 예산으로 돌리면 "우리 증강이 얼마나 기여했는가" 를 표로 낼 수 있습니다.

In [ ]:
# ═══════════════ 설정 ═══════════════   ← 실험할 때 여기만 바꾸세요
DATA_ROOT = r"D:/X"
#1. 팀 구글드라이브에 있는 PillData의 압축을 푼다. 
#2. 새로운 파일을 생성한다.
#3. 그 파일에 압축을 푼 파일을 넣고, 새로운 파일의 경로주소를 적는다.
# ex)D드라이브 안에 있는 X라는 이름의 파일에 PillData파일을 넣었다. 그럼 D:/X 로 설정

# ★ 02 의 증강 데이터셋. 증강 효과 비교용 기준선은 pill_yolo_base
DATA_YAML = f"{DATA_ROOT}/data/pill_yolo/data.yaml"
# DATA_YAML = f"{DATA_ROOT}/data/pill_yolo_base/data.yaml"

EXP_NAME = "exp_offline"
NOTE     = "기하증강 3배 + Copy&Paste(cropped_pills_review) / yolo11s / imgsz960"

MODEL    = "yolo11s.pt"      # yolo11n/s/m/l.pt, rtdetr-l.pt
EPOCHS   = 100               # 상한 (patience 로 조기 종료)
PATIENCE = 15                # ★ 팀 전체 통일
IMGSZ    = 960               # 각인 판독력에 직결. 01 의 박스 크기 분석 참고
BATCH    = 8                 # imgsz 960 이면 8~16.(전용 GPU메모리가 16GB일 시) OOM 이면 낮추세요
WORKERS  = 0                 # Windows 는 0 이 안전

# ---------- ★ 온라인 증강 ----------
# False = Ultralytics 내장 증강을 전부 끔 (02 오프라인 증강을 쓸 때)
# True  = 내장 증강 사용 (증강 없는 pill_yolo_base 로 실험할 때만)
USE_ONLINE_AUG = False

# USE_ONLINE_AUG=True 일 때만 적용되는 값
ON_DEGREES = 180.0           # 알약은 회전 불변 → 크게
ON_FLIPLR, ON_FLIPUD = 0.0, 0.0   # ★ 각인 때문에 0 권장
ON_HSV_H, ON_HSV_S, ON_HSV_V = 0.015, 0.5, 0.4
ON_SCALE = 0.5

# 두 경우 모두 적용
MOSAIC   = 0.5               # 알약이 여러 개 겹치는 상황 학습. 0.0 과 비교 실험 권장
TRANSLATE = 0.05
ERASING  = 0.0               # ★ 각인이 지워질 수 있어 항상 0

RESUME = False               # 세션이 끊긴 뒤 이어하기
SEED   = 42

import os
os.makedirs(f"{DATA_ROOT}/experiments", exist_ok=True)
print(f"실험    {EXP_NAME}")
print(f"변경점  {NOTE}")
print(f"데이터  {DATA_YAML}")
print(f"온라인 증강 {'ON — 내장 증강 사용' if USE_ONLINE_AUG else 'OFF — 02 오프라인 증강 사용'}")

In [ ]:
"""공통 유틸 — 이 셀을 먼저 실행하세요."""
import os, json, glob, csv, random
from collections import Counter, defaultdict

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont

random.seed(SEED); np.random.seed(SEED)


# ---------------------------------------------------------------- 한글 경로 IO
def imread_unicode(path):
    try:
        return cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        return None


def imwrite_unicode(path, img):
    ext = os.path.splitext(str(path))[1] or ".png"
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        return False
    buf.tofile(str(path)); return True


# ---------------------------------------------------------------- 한글 폰트
def find_korean_font():
    cands = ["C:/Windows/Fonts/malgun.ttf",
             "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
             "/System/Library/Fonts/AppleSDGothicNeo.ttc"]
    for pat in ("/usr/share/fonts/**/*CJK*.ttc", "/usr/share/fonts/**/*Nanum*.ttf",
                "/usr/share/fonts/**/*Gothic*.ttf"):
        cands += sorted(glob.glob(pat, recursive=True))
    for p in cands:
        if os.path.exists(p):
            try:
                ImageFont.truetype(p, 20); return p
            except Exception:
                pass
    return None

FONT_PATH = find_korean_font()
if FONT_PATH is None:
    print("⚠️  한글 폰트를 찾지 못했습니다. 라벨이 □□□ 로 보이면:")
    print("    Colab:   !apt-get install -y fonts-nanum && !fc-cache -fv")
else:
    print(f"한글 폰트: {FONT_PATH}")

_PALETTE = [(255,89,94),(56,176,0),(25,130,196),(255,202,58),(138,80,220),
            (0,187,249),(241,91,181),(155,200,60),(255,140,0),(0,200,170),
            (200,60,120),(120,160,255)]

def class_color(cid):
    return _PALETTE[int(cid) % len(_PALETTE)]


# ---------------------------------------------------------------- 시각화
def draw_detections(img_bgr, dets, id2name=None, font_scale=1.0,
                    box_thickness=3, show_conf=True):
    """★ 라벨 형식: "약이름 0.91"  (주석 뒤 한 칸 띄우고 confidence)"""
    img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    H, W = img_bgr.shape[:2]
    size = max(16, int(min(W, H) * 0.028 * font_scale))
    font = ImageFont.truetype(FONT_PATH, size) if FONT_PATH else ImageFont.load_default()

    for d in dets:
        cid = d["category_id"]
        x, y, w, h = [float(v) for v in d["bbox"]]
        color = class_color(cid)
        name = str(id2name.get(cid, cid)) if id2name else str(cid)
        label = f"{name} {d['score']:.2f}" if (show_conf and d.get("score") is not None) else name

        draw.rectangle([x, y, x + w, y + h], outline=color, width=box_thickness)
        tb = draw.textbbox((0, 0), label, font=font)
        tw, th = tb[2] - tb[0], tb[3] - tb[1]
        pad = max(3, size // 6)
        ly = y - th - pad * 2
        if ly < 0:
            ly = y + pad
        lx = min(max(0, x), W - tw - pad * 2)
        draw.rectangle([lx, ly, lx + tw + pad * 2, ly + th + pad * 2], fill=color)
        lum = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
        draw.text((lx + pad, ly + pad), label, font=font,
                  fill=(0, 0, 0) if lum > 150 else (255, 255, 255))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def save_per_image(records, img_dir, out_dir, id2name=None, conf_thr=0.0,
                   limit=0, suffix="", verbose=True):
    """★ 이미지 한 장당 결과 파일 한 개씩. 격자로 몰아넣지 않습니다."""
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for fn, dets in sorted(records.items()):
        p = os.path.join(img_dir, fn)
        if not os.path.exists(p):
            p = os.path.join(img_dir, os.path.basename(fn))
            if not os.path.exists(p):
                continue
        img = imread_unicode(p)
        if img is None:
            continue
        keep = [d for d in dets if d.get("score") is None or d["score"] >= conf_thr]
        vis = draw_detections(img, keep, id2name)
        stem, ext = os.path.splitext(os.path.basename(fn))
        imwrite_unicode(os.path.join(out_dir, f"{stem}{suffix}{ext or '.png'}"), vis)
        n += 1
        if limit and n >= limit:
            break
    if verbose:
        print(f"{n}장 저장 → {out_dir}  (이미지 1장 = 파일 1개)")
    return n


def show_image(path, max_width=900):
    from IPython.display import display
    im = Image.open(path)
    if im.width > max_width:
        im = im.resize((max_width, int(im.height * max_width / im.width)))
    display(im)


# ---------------------------------------------------------------- 매핑 로드
DATA_DIR  = os.path.dirname(DATA_YAML)
CMAP_JSON = os.path.join(DATA_DIR, "category_map.json")
if not os.path.exists(CMAP_JSON):
    raise SystemExit(f"{CMAP_JSON} 이 없습니다. 01 또는 02 를 먼저 실행하세요.")

CMAP = json.load(open(CMAP_JSON, encoding="utf-8"))
IDX2CAT = {int(k): int(v) for k, v in CMAP["idx2cat"].items()}   # YOLO 인덱스 → category_id
CAT2IDX = {int(k): int(v) for k, v in CMAP["cat2idx"].items()}
IDXNAME = {int(k): v for k, v in CMAP["names"].items()}          # YOLO 인덱스 → 이름
# ★ 주의: names 는 '인덱스' 로 키가 잡혀 있습니다. category_id 로 다시 매핑합니다.
ID2NAME = {IDX2CAT[i]: n for i, n in IDXNAME.items()}

print(f"\n클래스 {len(IDX2CAT)}종  YOLO 0~{max(IDX2CAT)} → category_id "
      f"{min(IDX2CAT.values())}~{max(IDX2CAT.values())}")
print(f"전처리 설정 (02 기록): {CMAP.get('preprocess')}")
print(f"증강 설정   (02 기록): {CMAP.get('augment')}")

---
## 1. 학습

In [ ]:
# !pip install ultralytics
from ultralytics import YOLO
from datetime import datetime

# ★ 온라인 증강 인자 구성
if USE_ONLINE_AUG:
    aug_kwargs = dict(
        degrees=ON_DEGREES, fliplr=ON_FLIPLR, flipud=ON_FLIPUD,
        hsv_h=ON_HSV_H, hsv_s=ON_HSV_S, hsv_v=ON_HSV_V, scale=ON_SCALE,
    )
else:
    # 02 에서 오프라인 증강을 마쳤으므로 내장 증강은 전부 끕니다
    aug_kwargs = dict(
        degrees=0.0, fliplr=0.0, flipud=0.0,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, scale=0.2,
    )

print("적용할 증강 인자:")
for k, v in aug_kwargs.items():
    print(f"  {k:<10} {v}")
print(f"  {'mosaic':<10} {MOSAIC}\n  {'erasing':<10} {ERASING}")

t0 = datetime.now()
model = YOLO(MODEL)
results = model.train(
    data=os.path.abspath(DATA_YAML),
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    workers=WORKERS, amp=True, resume=RESUME,
    mosaic=MOSAIC, translate=TRANSLATE, erasing=ERASING,
    mixup=0.0, copy_paste=0.0, shear=0.0, perspective=0.0,
    seed=SEED, deterministic=True,
    project=os.path.abspath(f"{DATA_ROOT}/runs"), name=EXP_NAME,
    plots=True, exist_ok=True,
    **aug_kwargs,
)
SAVE_DIR = str(results.save_dir)
print(f"\n소요 시간: {datetime.now() - t0}")
print(f"가중치: {SAVE_DIR}/weights/best.pt")

---
## 2. 실험 기록

**설정값과 결과를 자동으로 남깁니다.** 나중에 표로 만들 때 이 로그를 씁니다.
특히 `epochs_actual` (실제 정지 시점)이 모델 비교의 핵심 지표가 됩니다.

In [ ]:
rec = {
    "name": EXP_NAME, "note": NOTE,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "model": MODEL, "data": DATA_YAML,
    "epochs_set": EPOCHS, "patience": PATIENCE,
    "imgsz": IMGSZ, "batch": BATCH,
    "online_aug": USE_ONLINE_AUG,
    "augment": {**aug_kwargs, "mosaic": MOSAIC, "erasing": ERASING},
    # ★ 02 의 오프라인 증강 설정도 함께 기록 — 나중에 원인 추적이 가능해집니다
    "offline_aug": CMAP.get("augment"),
    "preprocess": CMAP.get("preprocess"),
    "save_dir": SAVE_DIR,
}
try:
    rec["metrics"] = {k: float(v) for k, v in results.results_dict.items()
                      if isinstance(v, (int, float))}
except Exception:
    pass

csv_path = f"{SAVE_DIR}/results.csv"
if os.path.exists(csv_path):
    with open(csv_path, encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))
    rec["epochs_actual"] = max(0, len(rows) - 1)

# 학습 데이터 구성도 기록 (증강 기여도 해석에 필요)
train_imgs = len(glob.glob(os.path.join(DATA_DIR, "images", "train", "*")))
rec["n_train_images"] = train_imgs

LOG_PATH = f"{DATA_ROOT}/experiments/log.jsonl"
with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"설정 {EPOCHS} 에포크 → 실제 {rec.get('epochs_actual','?')} 에포크에 종료")
print(f"학습 이미지 {train_imgs:,}장")
for k, v in rec.get("metrics", {}).items():
    if "mAP" in k or "precision" in k or "recall" in k:
        print(f"  {k:<28} {v:.4f}")
print(f"\n기록: {LOG_PATH}")

---
## 3. 학습 곡선

`results.png` 에서 확인할 것:
- **mAP 가 평평해진 지점** → 다음 실험에서 `EPOCHS` 를 줄여도 됨
- **train loss 는 내려가는데 val mAP 가 정체** → 과적합. 모델을 줄이거나 증강 강화
- **둘 다 계속 오르는 중** → `EPOCHS` 를 늘릴 여지

> 오프라인 증강을 많이 쓰면 **train loss 가 늦게 떨어지는 것이 정상**입니다.
> 같은 사진의 변형을 여러 번 보기 때문입니다. val mAP 를 기준으로 판단하세요.

In [ ]:
for f in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png", "R_curve.png"]:
    p = f"{SAVE_DIR}/{f}"
    if os.path.exists(p):
        print(f); show_image(p, max_width=950)

### 3-1. val 예측 확인 — **이미지 1장 = 파일 1개**

ultralytics 가 만드는 `val_batch*.jpg` 는 여러 장을 격자로 몰아넣어
알약과 라벨이 작아 확인이 어렵습니다.
여기서는 **한 장씩 따로** 그려서 저장합니다. 라벨은 `약이름 confidence` 형식입니다.

> `images/val` 의 이미지는 02 에서 **이미 전처리가 적용된 상태**로 저장되어 있습니다.
> 그래서 여기서는 추가 전처리 없이 그대로 예측합니다.
> (원본 폴더에서 직접 예측하려면 05 처럼 `preprocess` 를 먼저 걸어야 합니다.)

In [ ]:
import yaml

with open(DATA_YAML, encoding="utf-8") as f:
    dy = yaml.safe_load(f)

VAL_IMG_DIR = os.path.join(DATA_DIR, dy["val"])
PRED_DIR = f"{DATA_ROOT}/outputs/predictions/{EXP_NAME}_val"

best = YOLO(f"{SAVE_DIR}/weights/best.pt")
val_files = sorted(glob.glob(f"{VAL_IMG_DIR}/*"))[:12]
print(f"val 이미지 {len(glob.glob(f'{VAL_IMG_DIR}/*'))}장 중 {len(val_files)}장 시각화")

records = {}
for p in val_files:
    r = best.predict(p, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    dets = []
    if r.boxes is not None and len(r.boxes):
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        cls  = r.boxes.cls.cpu().numpy().astype(int)
        for j in range(len(xyxy)):
            x1, y1, x2, y2 = xyxy[j]
            dets.append({"category_id": IDX2CAT[int(cls[j])],
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                         "score": float(conf[j])})
    records[os.path.basename(p)] = dets

save_per_image(records, VAL_IMG_DIR, PRED_DIR, id2name=ID2NAME, suffix="_pred")

n_det = [len(v) for v in records.values()]
print(f"이미지당 검출 수  평균 {np.mean(n_det):.1f} / 최소 {min(n_det)} / 최대 {max(n_det)}")
print("★ 원본은 알약이 3~4개입니다. 평균이 이보다 크게 벗어나면 conf 나 학습을 점검하세요.")

for p in sorted(glob.glob(f"{PRED_DIR}/*"))[:3]:
    print("\n" + os.path.basename(p)); show_image(p)

---
## 4. 다음 실험 계획

**한 번에 하나만** 바꿔가며 위 셀을 다시 실행하세요.

| 실험 | 바꿀 것 | 가설 |
|---|---|---|
| `exp_base` | `pill_yolo_base`, `USE_ONLINE_AUG=False` | 바닥 성능 |
| `exp_online` | `pill_yolo_base`, `USE_ONLINE_AUG=True` | 온라인 증강만의 효과 |
| `exp_offline` | `pill_yolo` (기준선) | ★ 기하 3배 + Copy&Paste 효과 |
| `exp_imgsz640` | `IMGSZ` 960 → 640 | 각인 해상도가 정말 중요한지 |
| `exp_model_m` | `MODEL` s → m | 데이터가 충분하면 큰 모델이 유리 |
| `exp_nomosaic` | `MOSAIC` 0.5 → 0.0 | "최대 4개" 제약과 충돌 해소 |
| `exp_cp0` | 02 를 `N_SYNTH=0` 으로 재실행 | Copy&Paste 단독 기여도 분리 |

> 각 실험마다 `EXP_NAME` 과 `NOTE` 를 반드시 바꾸세요.
> 안 그러면 `runs/` 의 이전 결과를 덮어씁니다.

### 결과 해석 요령

| 관찰 | 해석 |
|---|---|
| `exp_offline` > `exp_online` | 오프라인 증강 설계(정책 차등 + Copy&Paste)가 유효했음 |
| `exp_offline` ≈ `exp_base` | 증강이 다양성을 못 만들고 있음 → `N_SYNTH` ↑, 크롭 다양성 ↑ |
| `exp_imgsz640` 이 크게 낮음 | 각인 판독이 병목 → `IMGSZ` 를 더 올리거나 2-stage 검토 |
| `exp_cp0` 만 낮음 | Copy&Paste 가 실제로 기여함 → 보고서 핵심 근거 |

### 다음 노트북
`04_evaluation.ipynb` — mAP, 오류 분석, 촬영 조건별 성능, 실험 비교표